# Data-Driven Predictive Maintenance Platform for Remaining Useful Life (RUL) Prediction
### Course: Project-Based Learning (PBL) | Department of AI & ML, TCET Mumbai
**Authors:** Yashavi Kharwar (07), Nikita Mishra (24), Satyam Mishra (25)  
**Guide:** Ms. Seema Nagrani, Assistant Professor of AI&ML  
**Benchmark:** NASA PHM08 Turbofan Engine Degradation Dataset (C-MAPSS Simulation)  

---

## 1. Project Objectives & Scope
- **RO1:** Analyse the PHM08 dataset and identify equipment degradation patterns across operational flight regimes.
- **RO2:** Develop predictive models (Baseline Random Forest and Sequence-based LSTM Network) to estimate Remaining Useful Life (RUL).
- **RO3:** Evaluate model performance using MAE, RMSE, and the asymmetric PHM08 Challenge Score under strict engine-wise separation (zero temporal leakage).
- **RO4:** Present predictions and degradation curves through an interactive decision-support dashboard.

In [ ]:
# Environment Setup & Verification
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")

## 2. Dataset Acquisition & Loading (Listing 1 in Report)
The NASA PHM08 dataset records multivariate time series representing turbofan degradation until failure under 6 operational flight regimes. Each observation contains:
- **Column 1:** Engine Unit ID
- **Column 2:** Operating Cycle (Time index)
- **Columns 3–5:** 3 Operational Settings (Altitude, Mach, Throttle Resolver Angle)
- **Columns 6–26:** 21 Sensor Measurements

In [ ]:
# Download dataset if running in Google Colab
data_path = Path("../data/train.txt")
if not data_path.exists():
    data_path = Path("data/train.txt")
if not data_path.exists():
    !python ../scripts/download_data.py || python scripts/download_data.py

cols = ["unit", "cycle"] + [f"op{i}" for i in range(1, 4)] + [f"s{i}" for i in range(1, 22)]
data = pd.read_csv(data_path, sep=r"\s+", header=None, names=cols)
print("Shape:", data.shape, "| Unique Engines:", data["unit"].nunique())
print("Missing Values:", data.isnull().sum().sum(), "| Duplicates:", data.duplicated().sum())
data.head()

## 3. Supervised RUL Target Generation (Listing 2 in Report)
For complete run-to-failure training engines, the remaining useful life at cycle $t$ is calculated as:
$$RUL(t) = T_f - t$$
Following established prognostics literature, we optionally apply a piecewise constant cap ($RUL_{cap} = 125$) early in the engine life when degradation is physically negligible.

In [ ]:
# RUL Label generation
max_cycles = data.groupby("unit")["cycle"].transform("max")
data["RUL_raw"] = max_cycles - data["cycle"]
data["RUL"] = data["RUL_raw"].clip(upper=125)

plt.figure(figsize=(10, 4))
for u in [1, 2, 3, 4, 5]:
    u_df = data[data["unit"] == u]
    plt.plot(u_df["cycle"], u_df["RUL"], label=f"Engine #{u}")
plt.xlabel("Operating Cycle")
plt.ylabel("Remaining Useful Life (Cycles)")
plt.title("Piecewise Linear RUL Target Profiles")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 4. Strict Engine-Wise Partitioning (Figure 5 in Report)
To avoid severe temporal data leakage, engines are split strictly by unit identifier:
- **Training:** 70% of engines (152 units)
- **Validation:** 15% of engines (33 units)
- **Testing:** 15% of engines (33 units)

*The StandardScaler is fitted exclusively on the 152 training engines.*

In [ ]:
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
units = data["unit"].unique()
np.random.shuffle(units)

n_train = int(len(units) * 0.70)
n_val = int(len(units) * 0.15)

train_units = units[:n_train]
val_units = units[n_train:n_train + n_val]
test_units = units[n_train + n_val:]

train_df = data[data["unit"].isin(train_units)].copy()
val_df = data[data["unit"].isin(val_units)].copy()
test_df = data[data["unit"].isin(test_units)].copy()

feature_cols = [f"op{i}" for i in range(1, 4)] + [f"s{i}" for i in range(1, 22)]

scaler = StandardScaler()
scaler.fit(train_df[feature_cols].values)

train_df[feature_cols] = scaler.transform(train_df[feature_cols].values)
val_df[feature_cols] = scaler.transform(val_df[feature_cols].values)
test_df[feature_cols] = scaler.transform(test_df[feature_cols].values)
print(f"Partitioned: {len(train_units)} Train, {len(val_units)} Val, {len(test_units)} Test units.")

## 5. Temporal Sequence Construction (Listing 3 & Figure 6)
Consecutive observations are grouped into temporal sliding windows of length $L = 30$ cycles:
$$X(t) = [x(t-L+1), \dots, x(t)] \in \mathbb{R}^{30 \times 24}$$

In [ ]:
def create_sequences(df, features, target="RUL", window=30):
    X, y = [], []
    for _, grp in df.groupby("unit"):
        grp_sorted = grp.sort_values("cycle")
        vals = grp_sorted[features].values
        targets = grp_sorted[target].values
        if len(vals) < window:
            pad = np.repeat(vals[:1], window - len(vals), axis=0)
            X.append(np.vstack([pad, vals]))
            y.append(targets[-1])
            continue
        for i in range(window, len(vals) + 1):
            X.append(vals[i - window:i])
            y.append(targets[i - 1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_train, y_train = create_sequences(train_df, feature_cols, window=30)
X_val, y_val = create_sequences(val_df, feature_cols, window=30)
X_test, y_test = create_sequences(test_df, feature_cols, window=30)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

## 6. Model Training & Benchmarking
We compare:
1. **Model A: Random Forest Baseline Regressor**
2. **Model B: Recurrent LSTM Neural Network** (Listing 4 in Report)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Model A: Baseline
X_train_flat = X_train.reshape(len(X_train), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

rf = RandomForestRegressor(n_estimators=100, max_depth=16, random_state=42, n_jobs=-1)
rf.fit(X_train_flat, y_train)
pred_rf = rf.predict(X_test_flat)

# PHM08 Asymmetric Score Function (Equation 13)
def phm_score(y_true, y_pred):
    d = y_pred - y_true
    d_clip = np.clip(d, -100, 70)
    s_early = np.sum(np.exp(-d_clip[d < 0] / 13.0) - 1.0)
    s_late = np.sum(np.exp(d_clip[d >= 0] / 10.0) - 1.0)
    return float(s_early + s_late)

print(f"Baseline Random Forest Evaluation:")
print(f"  MAE:  {mean_absolute_error(y_test, pred_rf):.2f} cycles")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, pred_rf)):.2f} cycles")
print(f"  Score: {phm_score(y_test, pred_rf):,.1f}")

## 7. Conclusions & Findings
1. Sensor signals in PHM08 display prominent drift during late wear cycles, enabling reliable RUL prognosis.
2. Strict engine-wise data partitioning ensures zero data leakage and reflects realistic fleet deployment conditions.
3. The asymmetric PHM08 scoring function appropriately penalizes hazardous late predictions compared to conservative early maintenance alerts.